# Chapter 9 — ConvNet Architecture Patterns

Maps to Chollet Ch.9. Three patterns turn a basic ConvNet into a modern one:
**residual connections**, **batch normalization**, **depthwise separable convolutions** — assembled into a
mini-Xception.

### The guiding idea: Modularity – Hierarchy – Reuse (MHR)
Good architectures = repeated **blocks** (modules), stacked into a deep **hierarchy**, **reusing** the same
pattern (e.g. conv reuses weights across positions). Deep stacks of *narrow* layers beat shallow stacks of
*wide* ones — up to a limit set by **vanishing gradients**, which residual connections fix.

In [ ]:
import os; os.environ["KERAS_BACKEND"]="tensorflow"
import keras, numpy as np
from keras import layers


## 1. Residual connections — fix vanishing gradients
Deep backprop is like the telephone game: gradient info degrades through many layers. **Fix:** add a block's
input back to its output (`x = add([block(x), x])`). This shortcut lets gradients flow noiselessly, enabling
arbitrarily deep nets (ResNet, 2015).

Catch: shapes must match to add. If a block changes channel count or downsamples, project the residual with a
**1×1 Conv2D** (and matching `strides`).

In [ ]:
def residual_block(x, filters, pooling=False):
    residual = x
    x = layers.Conv2D(filters, 3, activation="relu", padding="same")(x)
    x = layers.Conv2D(filters, 3, activation="relu", padding="same")(x)
    if pooling:
        x = layers.MaxPooling2D(2, padding="same")(x)
        residual = layers.Conv2D(filters, 1, strides=2)(residual)   # match downsample
    elif filters != residual.shape[-1]:
        residual = layers.Conv2D(filters, 1)(residual)              # match channels
    return layers.add([x, residual])

inputs = keras.Input(shape=(32, 32, 3))
x = layers.Rescaling(1./255)(inputs)
x = residual_block(x, 32,  pooling=True)
x = residual_block(x, 64,  pooling=True)
x = residual_block(x, 128, pooling=False)
x = layers.GlobalAveragePooling2D()(x)
outputs = layers.Dense(10, activation="softmax")(x)
res_model = keras.Model(inputs, outputs)
print("residual model params:", res_model.count_params())


## 2. Batch normalization — normalize *activations* between layers
`BatchNormalization` re-centers/re-scales each layer's outputs using batch statistics (and a moving average at
inference). It smooths gradient flow → faster, deeper training. Used heavily in ResNet/EfficientNet/Xception.

Best practice ordering: **Conv (no bias) → BatchNorm → Activation**. BN centers on zero, which is exactly the
pivot ReLU needs. (Drop the conv bias with `use_bias=False` — BN's shift replaces it.)
When **fine-tuning**, freeze BN layers (`layer.trainable=False`) so their running stats don't drift.

In [ ]:
def conv_bn_relu(x, filters, strides=1):
    x = layers.Conv2D(filters, 3, strides=strides, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    return x

inp = keras.Input((32,32,3))
y = conv_bn_relu(layers.Rescaling(1./255)(inp), 32)
y = layers.MaxPooling2D(2)(y)
y = conv_bn_relu(y, 64)
y = layers.GlobalAveragePooling2D()(y)
out = layers.Dense(10, activation="softmax")(y)
bn_model = keras.Model(inp, out)
print("conv->BN->relu block OK; params:", bn_model.count_params())


## 3. Depthwise separable convolutions — same power, far fewer params
`SeparableConv2D` splits a convolution into **depthwise** (one spatial filter per channel) + **pointwise**
(1×1 to mix channels). Assumes spatial and channel correlations are separable — usually true for images.
Result: fewer parameters, less compute, often *better* generalization. Basis of Xception/MobileNet.

In [ ]:
# Same in/out shape: compare parameter counts
probe = keras.Input((32,32,64))
std  = keras.Model(probe, layers.Conv2D(128, 3, padding="same")(probe))
sep  = keras.Model(probe, layers.SeparableConv2D(128, 3, padding="same")(probe))
print("standard Conv2D params   :", std.count_params())
print("SeparableConv2D params   :", sep.count_params())
print(f"reduction: {std.count_params()/sep.count_params():.1f}x fewer")


## 4. Putting it together — a mini-Xception
Combine all three: an entry stem, then repeated **SeparableConv → BN → ReLU** blocks each wrapped in a
**residual** connection with pooling. This is the template behind production image models.

In [ ]:
def mini_xception(input_shape=(32,32,3), num_classes=10):
    inputs = keras.Input(shape=input_shape)
    x = layers.Rescaling(1./255)(inputs)
    x = layers.Conv2D(32, 3, use_bias=False)(x)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    for size in [64, 128, 256]:
        residual = x
        x = layers.Activation("relu")(x)
        x = layers.SeparableConv2D(size, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.SeparableConv2D(size, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D(3, strides=2, padding="same")(x)
        residual = layers.Conv2D(size, 1, strides=2, padding="same", use_bias=False)(residual)
        x = layers.add([x, residual])
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs)

(Xtr, ytr), (Xte, yte) = keras.datasets.cifar10.load_data()
ytr, yte = ytr[:,0], yte[:,0]
Xtr, ytr, Xte, yte = Xtr[:10000], ytr[:10000], Xte[:2000], yte[:2000]

model = mini_xception()
model.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(Xtr, ytr, epochs=8, batch_size=64, validation_split=0.1, verbose=0)
print("mini-Xception CIFAR(subset) test acc:",
      round(model.evaluate(Xte, yte, verbose=0, return_dict=True)["accuracy"], 3))
# NOTE: small 10k-sample / 8-epoch subset for speed. On FULL CIFAR-10 (50k) for ~30 epochs with
# data augmentation, this same architecture reaches ~85-90%. The patterns scale; the tiny subset does not.


## 5. Beyond convolution: Vision Transformers (awareness)
Since ~2020, **Vision Transformers (ViT)** split an image into patches, embed them, and apply the Transformer
(Ch.15) — no convolution. With enough data they match/beat ConvNets. ConvNets still win on small data
(their translation-invariance prior is built in). For your contest: ConvNets are the safe default; know ViT
exists.

---
# ✍️ PROBLEMS

### P1 — Ablation study (residual on/off)
Train the §1 model **with** residual connections and a copy **without** the `add` (just the conv stack), both
deeper (5 blocks). Plot val-loss curves. Does the residual version train more stably / reach lower loss?

In [ ]:
# TODO


### P2 — BN ordering
Compare three blocks on CIFAR: (a) Conv(relu)→BN, (b) Conv(no bias)→BN→relu, (c) Conv(relu) only. Train each
5 epochs and compare val accuracy + training speed.

In [ ]:
# TODO


### P3 — Separable vs standard
Build two identical mini-ConvNets, one with `Conv2D`, one with `SeparableConv2D`. Compare param count, test
accuracy, and training time on CIFAR subset. Confirm separable is leaner with comparable accuracy.

In [ ]:
# TODO


### P4 — Scale the mini-Xception
Add a 4th block (size 512) and data augmentation (Ch.8). Train longer with EarlyStopping. How high can you
push CIFAR-10 test accuracy?

In [ ]:
# TODO


---
# 📋 TEMPLATES

### T1 — Residual block

In [ ]:
def residual_block(x, filters, pooling=False):
    residual = x
    x = layers.Conv2D(filters, 3, activation="relu", padding="same")(x)
    x = layers.Conv2D(filters, 3, activation="relu", padding="same")(x)
    if pooling:
        x = layers.MaxPooling2D(2, padding="same")(x)
        residual = layers.Conv2D(filters, 1, strides=2)(residual)
    elif filters != residual.shape[-1]:
        residual = layers.Conv2D(filters, 1)(residual)
    return layers.add([x, residual])


### T2 — Conv → BatchNorm → ReLU (correct ordering)

In [ ]:
def conv_bn_relu(x, filters, strides=1):
    x = layers.Conv2D(filters, 3, strides=strides, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    return layers.Activation("relu")(x)
# fine-tuning: freeze BN -> for l in base.layers: if isinstance(l, layers.BatchNormalization): l.trainable=False


### T3 — Mini-Xception (residual + BN + separable)

In [ ]:
def mini_xception(input_shape, num_classes):
    inputs = keras.Input(input_shape)
    x = layers.Rescaling(1./255)(inputs)
    x = layers.Conv2D(32, 3, use_bias=False)(x)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    for size in [64, 128, 256]:
        residual = x
        x = layers.Activation("relu")(x)
        x = layers.SeparableConv2D(size, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
        x = layers.SeparableConv2D(size, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D(3, strides=2, padding="same")(x)
        residual = layers.Conv2D(size, 1, strides=2, padding="same", use_bias=False)(residual)
        x = layers.add([x, residual])
    x = layers.GlobalAveragePooling2D()(x); x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs)


---
### ✅ Checklist
- [ ] Explain MHR and why deep-narrow beats shallow-wide (and the vanishing-gradient limit).
- [ ] Add residual connections, projecting with 1×1 conv when shapes change.
- [ ] Use BatchNorm correctly (Conv no-bias → BN → activation; freeze when fine-tuning).
- [ ] Swap in SeparableConv2D and explain the param/compute savings.
- [ ] Assemble a mini-Xception and train it.

**Next: Chapter 10** — *Interpreting what ConvNets learn*: visualizing intermediate activations, filters, and
class activation heatmaps (Grad-CAM). Say "Chapter 10".